# LAB 3 — Análise Qualitativa de Chunks para RAG

## Objetivos da Aula

1. **Entender os 4 critérios de qualidade de chunks**: coerência semântica, completude informacional, independência contextual e adequação ao modelo
2. **Implementar métricas objetivas** para avaliar chunks ANTES de indexá-los
3. **Comparar 3 estratégias de chunking** no contexto jurídico
4. **Evitar o problema: "Garbage In, Garbage Out"** — chunks ruins geram embeddings ruins, retrieval ruim e respostas ruins

## Diagrama: Por que analisar chunks qualitativamente?

```
┌─────────────────┐
│ Documento PDF   │  (ex: acórdão, lei, parecer)
└────────┬────────┘
         │
         ▼
┌─────────────────────────┐
│  CHUNKING (3 estratégias) │  ❌ SEM ANÁLISE → chunks ruins
│ Fixed/Recursive/Semantic │  ✅ COM ANÁLISE → chunks bons
└────────┬────────────────┘
         │
         ▼
┌──────────────────────────┐
│ Análise Qualitativa       │  ← VOCÊ ESTÁ AQUI
│ (4 critérios + métricas)  │     Evita 80% dos problemas RAG
└────────┬─────────────────┘
         │
         ▼
┌──────────────────┐
│ Embeddings (BGE) │
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Index (OpenSearch)│
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Retrieval + Gen  │
│ (Llama 3.1 8B)   │
└──────────────────┘
```

## Referência

LEWIS, P. et al. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks**. *arXiv preprint arXiv:2005.11401*, 2020.

GAO, Y. et al. **Retrieval-Augmented Generation for Large Language Models: A Survey**. *arXiv preprint arXiv:2312.10997*, 2023.

In [ ]:
# Célula Code 1: Instalação de dependências
# Este bloco instala TODAS as libs necessárias para análise qualitativa de chunks

import subprocess
import sys

# Lista de pacotes obrigatórios
packages = [
    'langchain',                           # Framework RAG principal
    'langchain-text-splitters',            # Splitters (Fixed, Recursive, Semantic)
    'langchain-experimental',              # Experimental splitters
    'sentence-transformers',               # Para embeddings BGE-M3 e análise
    'scikit-learn',                        # Cosine similarity e métricas ML
    'matplotlib',                          # Visualizações
    'seaborn',                             # Plots estatísticos
    'pandas',                              # DataFrames
    'numpy',                               # Operações numéricas
    'umap-learn',                          # Redução dimensional (UMAP)
    'nltk',                                # NLP utilities
]

# Instalar cada pacote silenciosamente
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✅ Todas as dependências instaladas com sucesso!")
print("\n📋 Versões instaladas:")

# Verificar versões dos pacotes críticos
import langchain
import sentence_transformers
import sklearn
import pandas as pd
import numpy as np
import umap

versions = {
    'langchain': langchain.__version__,
    'sentence-transformers': sentence_transformers.__version__,
    'scikit-learn': sklearn.__version__,
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'umap': umap.__version__,
}

for pkg, ver in versions.items():
    print(f"  {pkg}: {ver}")

## Seção Teórica: O que torna um chunk "bom"?

### 4 Critérios de Qualidade de Chunks

| Critério | Descrição | Impacto em RAG | Como Medir |
|----------|-----------|----------------|------------|
| **1. Coerência Semântica** | As sentenças do chunk tratam de um mesmo tema? | Embeddings mais focados | Cosine similarity entre embeddings de sentenças vizinhas |
| **2. Completude Informacional** | O chunk responde COMPLETAMENTE uma pergunta sobre seu tema? | Respostas mais precisas | Inspeção manual + análise de palavras-chave |
| **3. Independência Contextual** | O chunk faz sentido SEM os chunks anteriores/posteriores? | Menos confusion no LLM | Detecção de referências anafóricas e pronomes órfãos |
| **4. Adequação ao Modelo** | O tamanho do chunk é adequado ao embedding + LLM? | Melhor uso de context window | Análise de distribuição de tamanhos e tokens |

### Por que aplicar em contexto jurídico?

**Documentos jurídicos são ESTRUTURADOS**:
- Artigos, incisos, alíneas, parágrafos formam unidades lógicas
- Um chunk que corta no meio de um artigo é INÚTIL
- Referências cruzadas ("vide Art. 35") exigem contexto completo
- Jurisprudência tem "ementas" que devem estar completas

In [ ]:
# Célula Code 2: Definição do corpus jurídico para avaliação
# Criamos 3 documentos jurídicos REAIS (simulados) para testar nossos critérios

# Documento 1: Acórdão de Habeas Corpus (2000+ chars, 4 seções estruturadas)
acordao_hc = """
HABEAS CORPUS Nº 123.456/SP

EMENTA:
Habeas Corpus. Prisão preventiva. Fundamentação adequada. Presença de fumus comissi delicti e periculum libertatis. 
Incidência do art. 312 do Código de Processo Penal. Ordem denegada. 1. A jurisprudência do Superior Tribunal de 
Justiça é pacífica no sentido de que a prisão preventiva é medida excepcional que deve estar adequadamente 
fundamentada em decisão motivada. 2. Demonstrada a necessidade da prisão para a garantia da ordem pública, 
conveniência da instrução criminal e aplicação da lei penal, afasta-se a ilegalidade alegada. 3. Ordem denegada.

RELATÓRIO:
A reportagem de fatos é dispensada na forma do art. 38 da Lei 8.038/1990. Trata-se de habeas corpus impetrado 
contra decisão do Tribunal de Justiça do Estado de São Paulo que manteve a prisão preventiva decretada em desfavor 
de [nome do paciente], acusado da prática de crime previsto no art. 33 da Lei nº 11.343/2006 (Lei de Drogas). 
O paciente sustenta ser ilegítima a custódia cautelar ante a falta de fundamentação adequada e a impossibilidade de 
satisfação dos pressupostos legais.

FUNDAMENTAÇÃO:
A prisão preventiva é exceção no ordenamento jurídico brasileiro. Contudo, quando presente o fumus comissi delicti 
e o periculum libertatis, revela-se como medida legítima e necessária à garantia da ordem pública e da aplicação 
da lei penal, consoante previsão do art. 312 do Código de Processo Penal. A decisão que a decretou encontra-se 
adequadamente fundamentada. Demonstra a presença dos requisitos legais para sua manutenção. Não há que se falar 
em ilegalidade. A matéria alegada não enseja o reconhecimento de direito líquido e certo ao paciente.

DISPOSITIVO:
Pelo exposto, DENEGO a ordem de habeas corpus. A decisão monocrática será proferida na forma do regimento interno.
"""

# Documento 2: Artigos da Lei nº 11.343/2006 (Lei de Drogas) com estrutura jurídica real
lei_11343 = """
LEI Nº 11.343, DE 23 DE AGOSTO DE 2006

Institui o Sistema Nacional de Políticas Públicas sobre Drogas - SISNAD; prescreve medidas para prevenção do uso 
indevido, atenção e reinserção social de usuários e dependentes de drogas; estabelece normas para repressão à 
produção não autorizada e ao tráfico ilícito de drogas; define crimes e dá outras providências.

Art. 28. Quem adquirir, guardar, tiver em depósito, transportar ou trouxer consigo, para consumo pessoal, drogas 
sem autorização ou em desacordo com determinação legal ou regulamentar será submetido às seguintes penas: 
I - advertência sobre os efeitos das drogas; 
II - prestação de serviços à comunidade; 
III - medida educativa de comparecimento a programa ou curso educativo.

§ 1º As penas previstas nos incisos II e III do caput deste artigo serão aplicadas observando-se as normas do art. 
76 da Lei nº 9.099, de 26 de setembro de 1995, que estabelece normas gerais aplicáveis à ação penal de iniciativa 
pública, dependente de representação ou privada, nos crimes de menor potencial ofensivo, no que couber, sendo certo 
que será sempre admitida a substituição alternativa ou cumulativamente de qualquer das penas por multa.

Art. 33. Importar, exportar, remeter, preparar, produzir, fabricar, adquirir, vender, distribuir, entregar a qualquer 
título, guardar, ter em depósito, transportar, trazer consigo, ainda que gratuitamente, sem autorização ou em 
desacordo com determinação legal ou regulamentar, matéria-prima, insumo ou produto químico destinado à elaboração 
de drogas ilícitas; ativar, trabalhar ou agir de qualquer forma para o incremento, produção ou tráfico ilícito de 
drogas: Pena - reclusão de 5 a 15 anos e pagamento de 500 a 1.500 salários mínimos.

§ 4º nos crimes definidos no caput e nos §§ 1º e 3º deste artigo, as penas poderão ser reduzidas de um sexto a dois 
terços, vedada a substituição por penas restritivas de direitos, desde que o agente colabore efetivamente com a 
justiça na investigação e processuais penais, mediante a confissão circunstanciada da participação na atividade 
criminosa.
"""

# Documento 3: Relatório de Inteligência Policial com tabela e análise
relatorio_inteligencia = """
RELATÓRIO DE INTELIGÊNCIA - 1º TRIMESTRE 2025
Secretaria de Segurança Pública do Estado de São Paulo
Departamento de Análise Criminal

RESUMO EXECUTIVO:
O presente relatório apresenta análise consolidada das ocorrências registradas no período de janeiro a março de 2025. 
Os dados indicam aumento de 12,3% em roubos a mão armada e redução de 8,7% em furtos quando comparado ao trimestre 
anterior. As operações de inteligência resultaram em 234 prisões relacionadas ao tráfico de drogas.

ANÁLISE POR TIPO DE CRIME:
Crime | Jan/2025 | Fev/2025 | Mar/2025 | Total | Variação
Homicídio doloso | 156 | 142 | 148 | 446 | -5,2%
Roubo a mão armada | 523 | 587 | 612 | 1.722 | +12,3%
Furto simples | 1.234 | 1.189 | 1.127 | 3.550 | -8,7%
Tráfico de drogas | 89 | 95 | 76 | 260 | -14,6%
Sequestro | 3 | 2 | 1 | 6 | -66,7%

OPERAÇÕES ESPECIAIS:
Durante o período, foram deflagradas 18 operações estruturadas com envolvimento de 45 agentes de inteligência. 
Destaca-se a Operação "Hércules" de combate ao tráfico de cocaína na zona leste, que resultou na apreensão de 
234 kg de cocaína e 456 kg de maconha. As informações coletadas em campo foram consolidadas em banco de dados 
inteligência conforme protocolos da Resolução SENASP nº 1/2009.
"""

# Dicionário com os 3 documentos
corpus = {
    'acordao_hc_001.txt': acordao_hc,
    'lei_11343_excerpts.txt': lei_11343,
    'relatorio_inteligencia_q1_2025.txt': relatorio_inteligencia,
}

print("✅ Corpus jurídico definido com 3 documentos variados:\n")
for nome, conteudo in corpus.items():
    print(f"  📄 {nome:<40} {len(conteudo):>6} caracteres")

print(f"\n📊 Total no corpus: {sum(len(v) for v in corpus.values())} caracteres")

## Seção: Preparando as 3 Estratégias de Chunking

Vamos implementar 3 estratégias diferentes e depois avaliar a QUALIDADE de cada uma:

1. **Fixed Chunking**: chunks de tamanho fixo (800 chars) com overlap (150 chars)
2. **Recursive Chunking**: separa por delimitadores jurídicos (artigo, inciso, etc.) recursivamente
3. **Semantic Chunking**: usa embeddings para agrupar chunks semanticamente coerentes

In [ ]:
# Célula Code 3: Implementar as 3 estratégias de chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain.schema import Document
import re

# Configuração comum para todos os splitters
CHUNK_SIZE = 800      # Tamanho alvo de cada chunk em caracteres
CHUNK_OVERLAP = 150   # Sobreposição para manter contexto

# ========== ESTRATÉGIA 1: FIXED CHUNKING ==========
def chunking_fixed(texto, doc_name):
    """
    Divide o texto em chunks de tamanho FIXO com overlap.
    Simples, previsível, mas pode cortar no meio de conceitos importantes.
    """
    # CharacterTextSplitter: divide por um separador único (\n\n)
    splitter = CharacterTextSplitter(
        separator="\n\n",                    # Separa por parágrafos
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,                   # Usa tamanho em caracteres
    )
    
    chunks = splitter.split_text(texto)
    
    # Converter para Documents com metadados
    documents = [
        Document(
            page_content=chunk,
            metadata={
                'source': doc_name,
                'strategy': 'FIXED',
                'chunk_size_chars': len(chunk),
                'chunk_id': i,
            }
        )
        for i, chunk in enumerate(chunks)
    ]
    
    return documents

# ========== ESTRATÉGIA 2: RECURSIVE CHUNKING ==========
def chunking_recursive(texto, doc_name):
    """
    Divide recursivamente usando separadores jurídicos customizados.
    Respeita estrutura de artigos, incisos, etc. Muito melhor para direito!
    """
    # Separadores em ordem de preferência (tenta manter estrutura jurídica)
    separadores = [
        "\n\nArt\.",          # Separador preferido: novo artigo
        "\n\n§",             # Novo parágrafo dentro de artigo
        "\nI -",             # Inciso
        "\nII -",
        "\n",                # Quebra de linha simples
        " ",                 # Espaço (último recurso)
    ]
    
    splitter = RecursiveCharacterTextSplitter(
        separators=separadores,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    
    chunks = splitter.split_text(texto)
    
    documents = [
        Document(
            page_content=chunk,
            metadata={
                'source': doc_name,
                'strategy': 'RECURSIVE',
                'chunk_size_chars': len(chunk),
                'chunk_id': i,
            }
        )
        for i, chunk in enumerate(chunks)
    ]
    
    return documents

# ========== ESTRATÉGIA 3: SEMANTIC CHUNKING (SIMPLIFICADA) ==========
def chunking_semantic_simple(texto, doc_name):
    """
    Versão simplificada: agrupa parágrafos semanticamente similares.
    (Versão FULL usaria embeddings, aqui usamos heurísticas de keywords)
    """
    # Primeiro, aplica recursive chunking como base
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " "],
        chunk_size=400,  # Chunks menores inicialmente
        chunk_overlap=100,
        length_function=len,
    )
    
    sub_chunks = splitter.split_text(texto)
    
    # Pseudo-agrupamento: agrupa chunks que falam do mesmo "tópico"
    # (Em produção, usaríamos cosine similarity de embeddings)
    merged_chunks = []
    current_merge = ""
    
    for chunk in sub_chunks:
        # Se chunk+current fica menor que CHUNK_SIZE, mescla
        if len(current_merge) + len(chunk) + 10 < CHUNK_SIZE:
            current_merge += "\n\n" + chunk if current_merge else chunk
        else:
            # Termina merge atual e começa novo
            if current_merge:
                merged_chunks.append(current_merge)
            current_merge = chunk
    
    if current_merge:
        merged_chunks.append(current_merge)
    
    documents = [
        Document(
            page_content=chunk,
            metadata={
                'source': doc_name,
                'strategy': 'SEMANTIC',
                'chunk_size_chars': len(chunk),
                'chunk_id': i,
            }
        )
        for i, chunk in enumerate(merged_chunks)
    ]
    
    return documents

# ========== APLICAR AS 3 ESTRATÉGIAS AO CORPUS ==========
chunks_por_estrategia = {
    'FIXED': [],
    'RECURSIVE': [],
    'SEMANTIC': [],
}

# Iterar sobre cada documento do corpus
for doc_name, texto in corpus.items():
    # Aplicar 3 estratégias
    chunks_fixed = chunking_fixed(texto, doc_name)
    chunks_recursive = chunking_recursive(texto, doc_name)
    chunks_semantic = chunking_semantic_simple(texto, doc_name)
    
    # Armazenar
    chunks_por_estrategia['FIXED'].extend(chunks_fixed)
    chunks_por_estrategia['RECURSIVE'].extend(chunks_recursive)
    chunks_por_estrategia['SEMANTIC'].extend(chunks_semantic)

# Exibir resumo
print("✅ Chunking completo! Resumo por estratégia:\n")
for estrategia, chunks in chunks_por_estrategia.items():
    total_chars = sum(len(c.page_content) for c in chunks)
    avg_size = total_chars / len(chunks) if chunks else 0
    print(f"  {estrategia:<12} | {len(chunks):>3} chunks | {total_chars:>6} chars | avg {avg_size:.0f} chars/chunk")

## Seção: Critério 1 — Coerência Semântica

Um chunk **coerente** tem sentenças que tratam do mesmo tema. Medimos isso calculando a similaridade cosseno entre embeddings de sentenças vizinhas.

- **Score alto (0.7+)**: sentenças tratam do mesmo tema
- **Score baixo (<0.4)**: salto temático (red flag!)

In [ ]:
# Célula Code 4: Calcular coerência semântica de cada chunk

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

# Download de tokenizer NLTK
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Carregar modelo de embeddings (lightweight para Colab)
print("🔄 Carregando modelo de embeddings (paraphrase-multilingual-MiniLM-L12-v2)...")
model_embedding = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Modelo carregado!\n")

def calcular_coerencia_chunk(chunk_text):
    """
    Calcula score de coerência semântica de um chunk.
    
    Método:
    1. Dividir chunk em sentenças
    2. Gerar embedding para cada sentença
    3. Calcular cosine similarity entre sentenças ADJACENTES
    4. Retornar média das similarities (0-1, 1 = muito coerente)
    """
    # Passo 1: Tokenizar em sentenças
    sentencas = sent_tokenize(chunk_text)
    
    # Se muito poucas sentenças, impossível medir coerência
    if len(sentencas) < 2:
        return 0.5  # Score neutro
    
    # Passo 2: Gerar embeddings
    embeddings = model_embedding.encode(sentencas, convert_to_numpy=True)
    
    # Passo 3: Calcular similarities entre sentenças ADJACENTES
    similarities = []
    for i in range(len(embeddings) - 1):
        # Cosine similarity entre sentença i e i+1
        sim = cosine_similarity(
            [embeddings[i]],
            [embeddings[i+1]]
        )[0][0]
        similarities.append(float(sim))
    
    # Passo 4: Retornar média
    coerencia = sum(similarities) / len(similarities) if similarities else 0.5
    return coerencia

# Aplicar a cada chunk de cada estratégia
print("📊 Calculando coerência semântica para todos os chunks...\n")

resultados_coerencia = []

for estrategia, chunks in chunks_por_estrategia.items():
    print(f"  {estrategia}...", end="")
    
    for chunk in chunks:
        coerencia = calcular_coerencia_chunk(chunk.page_content)
        
        resultados_coerencia.append({
            'estrategia': estrategia,
            'fonte': chunk.metadata['source'],
            'chunk_id': chunk.metadata['chunk_id'],
            'coerencia': coerencia,
            'tamanho_chars': len(chunk.page_content),
            'preview': chunk.page_content[:80].replace('\n', ' '),
        })
    
    print(f" ✅")

# Converter para DataFrame
df_coerencia = pd.DataFrame(resultados_coerencia)

print(f"\n✅ Cálculos completos! {len(df_coerencia)} chunks avaliados.\n")

# Análise descritiva por estratégia
print("📈 Resumo de Coerência por Estratégia:\n")
resume_coerencia = df_coerencia.groupby('estrategia')['coerencia'].agg([
    ('avg', 'mean'),
    ('min', 'min'),
    ('max', 'max'),
    ('std', 'std'),
    ('count', 'count'),
])
print(resume_coerencia.round(3))

# Top-3 chunks mais coerentes e menos coerentes por estratégia
print("\n🏆 Top-3 chunks MAIS COERENTES (por estratégia):\n")
for est in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    df_est = df_coerencia[df_coerencia['estrategia'] == est]
    top3 = df_est.nlargest(3, 'coerencia')
    print(f"  {est}:")
    for idx, row in top3.iterrows():
        print(f"    - Coerência {row['coerencia']:.3f} | \"{row['preview'][:50]}...\"")

print("\n⚠️  Top-3 chunks MENOS COERENTES (por estratégia):\n")
for est in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    df_est = df_coerencia[df_coerencia['estrategia'] == est]
    bottom3 = df_est.nsmallest(3, 'coerencia')
    print(f"  {est}:")
    for idx, row in bottom3.iterrows():
        print(f"    - Coerência {row['coerencia']:.3f} | \"{row['preview'][:50]}...\"")

## Seção: Critério 2 — Completude Informacional (Análise Manual Guiada)

Um chunk tem **completude** se responde COMPLETAMENTE uma pergunta sobre seu tema, sem deixar informações críticas em chunks anteriores/posteriores.

Esta análise é **semi-manual** porque embeddings não conseguem medir "completude" de forma confiável.

In [ ]:
# Célula Code 5: Framework de inspeção manual para completude informacional

def avaliar_chunk_manual(chunk_text, query_simulada):
    """
    Avalia completude de um chunk em relação a uma query simulada.
    
    Retorna dict com:
      - contém_fato_principal: boolean
      - contém_contexto_suficiente: boolean
      - frase_cortada_no_inicio: boolean
      - frase_cortada_no_fim: boolean
      - score_completude: 0-1
    """
    resultado = {
        'query': query_simulada,
        'contém_fato_principal': True,  # Simplificado para demo
        'contém_contexto': True,
        'frase_cortada_inicio': False,
        'frase_cortada_fim': False,
        'score': 1.0,
    }
    
    # Heurística 1: Chunk começa com pronome ou artigo orfão?
    primeiras_palavras = chunk_text.strip().split()[:3]
    orfaos = ['ele', 'ela', 'eles', 'isso', 'aquele', 'este', 'esse', 'o', 'a', 'os', 'as']
    
    if primeiras_palavras and primeiras_palavras[0].lower() in orfaos:
        resultado['frase_cortada_inicio'] = True
        resultado['score'] -= 0.2
    
    # Heurística 2: Chunk termina abruptamente (sem ponto final)?
    if not chunk_text.strip().endswith(('.', '!', '?', '"', 'incisos.', 'alínea.')):
        resultado['frase_cortada_fim'] = True
        resultado['score'] -= 0.1
    
    # Heurística 3: Contém keywords relacionadas à query?
    palavras_query = set(query_simulada.lower().split())
    palavras_chunk = set(chunk_text.lower().split())
    overlap = len(palavras_query & palavras_chunk)
    
    if overlap < len(palavras_query) * 0.3:
        resultado['contém_fato_principal'] = False
        resultado['score'] -= 0.15
    
    resultado['score'] = max(0.0, min(1.0, resultado['score']))
    return resultado

# Definir queries jurídicas REAIS para testar
queries_teste = [
    "O que é prisão preventiva?",
    "Quais são as penas para tráfico de drogas?",
    "Qual foi o número de homicídios em janeiro de 2025?",
    "Como funciona a redução de pena por colaboração?",
    "O que mudou nas ocorrências de roubo no Q1 2025?",
]

# Selecionar 5 chunks de CADA estratégia para avaliação manual
resultados_completude = []

for estrategia in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    chunks = chunks_por_estrategia[estrategia]
    
    # Pegar primeiros 5 chunks de cada estratégia (amostragem)
    chunks_amostra = chunks[:5]
    
    for i, chunk in enumerate(chunks_amostra):
        # Para cada chunk, aplicar uma query aleatória
        query = queries_teste[i % len(queries_teste)]
        
        avaliacao = avaliar_chunk_manual(chunk.page_content, query)
        
        resultados_completude.append({
            'estrategia': estrategia,
            'chunk_id': i,
            'query': query,
            'score_completude': avaliacao['score'],
            'frase_cortada_inicio': avaliacao['frase_cortada_inicio'],
            'frase_cortada_fim': avaliacao['frase_cortada_fim'],
        })

df_completude = pd.DataFrame(resultados_completude)

print("✅ Análise de completude completa! Resumo por estratégia:\n")
resume_completude = df_completude.groupby('estrategia')['score_completude'].agg([
    ('avg', 'mean'),
    ('chunks_com_corte_inicio', lambda x: (df_completude.loc[x.index, 'frase_cortada_inicio']).sum()),
    ('chunks_com_corte_fim', lambda x: (df_completude.loc[x.index, 'frase_cortada_fim']).sum()),
    ('total', 'count'),
])
print(resume_completude)

print("\n📋 Tabela completa de avaliação manual:\n")
print(df_completude.to_string())

## Seção: Critério 3 — Independência Contextual

Um chunk tem **independência contextual** se faz sentido SEM os chunks anteriores/posteriores.

Detectamos "chunks órfãos" procurando por:
- Começam com pronomes (ele, ela, isso, aquele, este)
- Começam com artigos sem antecedente
- Começam com expressões anafóricas ("Conforme mencionado", "Como vimos")

In [ ]:
# Célula Code 6: Testar independência contextual e detectar chunks órfãos

def detectar_chunk_orfao(texto):
    """
    Detecta se um chunk é "órfão" (não faz sentido sem contexto).
    
    Retorna: (é_orfão: bool, justificativa: str)
    """
    # Padrões que indicam chunk órfão
    padroes_orfaos = [
        (r'^\s*(ele|ela|eles|elas|isso|aquele|aquela|este|esse)\s', 'Começa com pronome órfão'),
        (r'^\s*[Cc]onforme mencionado', 'Referência a seção anterior'),
        (r'^\s*[Cc]omo vimos', 'Referência a seção anterior'),
        (r'^\s*[Aa] partir disso', 'Dependência contextual'),
        (r'^\s*[Pp]or outro lado', 'Contraste implícito com anterior'),
        (r'^\s*[Oo]u seja', 'Continuação de raciocínio'),
        (r'^\s*[Pp]or lo tanto', 'Conclusão de argumento anterior'),
    ]
    
    # Testar cada padrão
    for padrao, razao in padroes_orfaos:
        if re.search(padrao, texto):
            return True, razao
    
    return False, "Chunk independente"

# Aplicar a TODOS os chunks
resultados_independencia = []

for estrategia, chunks in chunks_por_estrategia.items():
    total_chunks = len(chunks)
    chunks_orfaos = 0
    
    for chunk in chunks:
        eh_orfao, justificativa = detectar_chunk_orfao(chunk.page_content)
        
        if eh_orfao:
            chunks_orfaos += 1
        
        resultados_independencia.append({
            'estrategia': estrategia,
            'chunk_id': chunk.metadata['chunk_id'],
            'fonte': chunk.metadata['source'],
            'eh_orfao': eh_orfao,
            'justificativa': justificativa,
            'tamanho': len(chunk.page_content),
            'preview': chunk.page_content[:60].replace('\n', ' '),
        })
    
    pct_orfaos = (chunks_orfaos / total_chunks * 100) if total_chunks > 0 else 0
    print(f"{estrategia:<12} | {chunks_orfaos:>2}/{total_chunks:>2} chunks órfãos ({pct_orfaos:>5.1f}%)")

df_independencia = pd.DataFrame(resultados_independencia)

print("\n✅ Análise de independência completa!\n")

# Mostrar exemplos de chunks órfãos
chunks_orfaos = df_independencia[df_independencia['eh_orfao']]
if len(chunks_orfaos) > 0:
    print("⚠️  Exemplos de chunks órfãos (dependentes de contexto anterior):\n")
    for idx, row in chunks_orfaos.head(5).iterrows():
        print(f"  {row['estrategia']} | {row['justificativa']}")
        print(f"    \"{row['preview'][:60]}...\"\n")
else:
    print("✅ Nenhum chunk órfão detectado!")

## Seção: Critério 4 — Distribuição de Tamanhos e Adequação ao Modelo

In [ ]:
# Célula Code 7: Análise estatística de tamanhos de chunks

import matplotlib.pyplot as plt
import seaborn as sns

# Preparar dados de tamanhos
resultados_tamanhos = []

for estrategia, chunks in chunks_por_estrategia.items():
    tamanhos = [len(chunk.page_content) for chunk in chunks]
    tokens_estimado = [len(chunk.page_content) / 4 for chunk in chunks]  # Estimativa: 1 token ≈ 4 chars
    
    # Calcular métricas
    mean_size = sum(tamanhos) / len(tamanhos) if tamanhos else 0
    min_size = min(tamanhos) if tamanhos else 0
    max_size = max(tamanhos) if tamanhos else 0
    std_dev = (sum((x - mean_size) ** 2 for x in tamanhos) / len(tamanhos)) ** 0.5 if tamanhos else 0
    
    # Coeficiente de variação (CV)
    cv = (std_dev / mean_size) if mean_size > 0 else 0
    
    # Percentual de chunks acima de 1024 tokens (limite de alguns embeddings)
    pct_acima_1024 = sum(1 for t in tokens_estimado if t > 1024) / len(tokens_estimado) * 100 if tokens_estimado else 0
    
    # Percentual de chunks "lixo" (< 50 chars)
    pct_lixo = sum(1 for t in tamanhos if t < 50) / len(tamanhos) * 100 if tamanhos else 0
    
    resultados_tamanhos.append({
        'estrategia': estrategia,
        'media_chars': mean_size,
        'min_chars': min_size,
        'max_chars': max_size,
        'std_dev': std_dev,
        'cv': cv,
        'pct_acima_1024_tokens': pct_acima_1024,
        'pct_lixo': pct_lixo,
        'total_chunks': len(chunks),
    })

df_tamanhos = pd.DataFrame(resultados_tamanhos)

print("📊 Análise estatística de tamanhos por estratégia:\n")
print(df_tamanhos[['estrategia', 'media_chars', 'min_chars', 'max_chars', 'std_dev', 'cv']].to_string(index=False))

print("\n⚠️  Métricas de qualidade:")
print(df_tamanhos[['estrategia', 'pct_acima_1024_tokens', 'pct_lixo']].to_string(index=False))

print("\n💡 Interpretação do Coeficiente de Variação (CV):")
print("  CV < 0.3: Tamanhos muito consistentes (bom!)")
print("  CV 0.3-0.6: Variação moderada (aceitável)")
print("  CV > 0.6: Tamanhos muito inconsistentes (problema!)\n")

# Criar visualização
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análise de Distribuição de Tamanhos de Chunks', fontsize=16, fontweight='bold')

# Subplot 1: Histograma de tamanhos
ax1 = axes[0, 0]
for estrategia in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    tamanhos = [len(chunk.page_content) for chunk in chunks_por_estrategia[estrategia]]
    ax1.hist(tamanhos, alpha=0.6, label=estrategia, bins=15)
ax1.set_xlabel('Tamanho do chunk (caracteres)')
ax1.set_ylabel('Frequência')
ax1.set_title('Distribuição de tamanhos')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Subplot 2: Boxplot
ax2 = axes[0, 1]
data_boxplot = []
labels_boxplot = []
for est in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    tamanhos = [len(chunk.page_content) for chunk in chunks_por_estrategia[est]]
    data_boxplot.append(tamanhos)
    labels_boxplot.append(est)
ax2.boxplot(data_boxplot, labels=labels_boxplot)
ax2.set_ylabel('Tamanho (caracteres)')
ax2.set_title('Boxplot: Distribuição de tamanhos')
ax2.grid(True, alpha=0.3)

# Subplot 3: Coeficiente de Variação (CV)
ax3 = axes[1, 0]
ax3.bar(df_tamanhos['estrategia'], df_tamanhos['cv'], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax3.axhline(y=0.3, color='r', linestyle='--', label='CV=0.3 (limite bom)', alpha=0.7)
ax3.axhline(y=0.6, color='orange', linestyle='--', label='CV=0.6 (limite aceitável)', alpha=0.7)
ax3.set_ylabel('Coeficiente de Variação')
ax3.set_title('Consistência de tamanhos (CV baixo = melhor)')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Subplot 4: Chunks "problemáticos"
ax4 = axes[1, 1]
data_problematico = [
    df_tamanhos.loc[df_tamanhos['estrategia'] == 'FIXED', 'pct_lixo'].values[0],
    df_tamanhos.loc[df_tamanhos['estrategia'] == 'RECURSIVE', 'pct_lixo'].values[0],
    df_tamanhos.loc[df_tamanhos['estrategia'] == 'SEMANTIC', 'pct_lixo'].values[0],
]
ax4.bar(['FIXED', 'RECURSIVE', 'SEMANTIC'], data_problematico, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax4.set_ylabel('% de chunks')
ax4.set_title('Chunks "lixo" (< 50 chars) - menor é melhor')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/analise_tamanhos_chunks.png', dpi=100, bbox_inches='tight')
print("✅ Gráfico salvo em /tmp/analise_tamanhos_chunks.png")
plt.show()

## Seção: Score Agregado de Qualidade

In [ ]:
# Célula Code 8: Calcular Score Final de Qualidade para cada estratégia

# Agregando os 4 critérios em um SCORE FINAL (0-10)

df_score_final = pd.DataFrame()

for estrategia in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    # Critério 1: Coerência Semântica (peso 0.35)
    coerencia_avg = df_coerencia[df_coerencia['estrategia'] == estrategia]['coerencia'].mean()
    score_coerencia = coerencia_avg  # Já está em 0-1
    
    # Critério 2: Completude Informacional (peso 0.15)
    if len(df_completude[df_completude['estrategia'] == estrategia]) > 0:
        completude_avg = df_completude[df_completude['estrategia'] == estrategia]['score_completude'].mean()
    else:
        completude_avg = 0.5
    score_completude = completude_avg
    
    # Critério 3: Independência Contextual (peso 0.30)
    chunks_estrategia = chunks_por_estrategia[estrategia]
    total_chunks = len(chunks_estrategia)
    chunks_orfaos = len(df_independencia[
        (df_independencia['estrategia'] == estrategia) & (df_independencia['eh_orfao'])
    ])
    pct_orfaos = (chunks_orfaos / total_chunks) if total_chunks > 0 else 0
    score_independencia = 1 - pct_orfaos  # Inverso: menos órfãos = melhor
    
    # Critério 4: Adequação ao Modelo (peso 0.20)
    df_tam_est = df_tamanhos[df_tamanhos['estrategia'] == estrategia]
    cv = df_tam_est['cv'].values[0]
    # CV baixo é melhor: penalizar CV alto
    score_tamanho = 1 - min(cv, 1.0)  # Normalizar
    
    # SCORE FINAL (ponderado)
    score_final_normalizado = (
        (score_coerencia * 0.35) +
        (score_independencia * 0.30) +
        (score_tamanho * 0.20) +
        (score_completude * 0.15)
    )
    
    # Converter para escala 0-10
    score_final_10 = score_final_normalizado * 10
    
    # Armazenar
    df_score_final = pd.concat([df_score_final, pd.DataFrame({
        'estrategia': [estrategia],
        'coerencia (0.35)': [round(score_coerencia, 3)],
        'independencia (0.30)': [round(score_independencia, 3)],
        'tamanho (0.20)': [round(score_tamanho, 3)],
        'completude (0.15)': [round(score_completude, 3)],
        'score_final': [round(score_final_normalizado, 3)],
        'score_10': [round(score_final_10, 2)],
    })], ignore_index=True)

print("🏆 SCORE FINAL DE QUALIDADE (0-10):\n")
print(df_score_final[['estrategia', 'score_10']].to_string(index=False))

print("\n📊 Detalhamento dos 4 critérios:\n")
print(df_score_final[[
    'estrategia',
    'coerencia (0.35)',
    'independencia (0.30)',
    'tamanho (0.20)',
    'completude (0.15)',
    'score_final'
]].to_string(index=False))

# Gerar recomendação automática
melhor_estrategia = df_score_final.loc[df_score_final['score_10'].idxmax()]
print(f"\n✅ RECOMENDAÇÃO: Use estratégia {melhor_estrategia['estrategia']} (score {melhor_estrategia['score_10']}/10)")

## Seção: Análise Específica para Contexto Jurídico Brasileiro

In [ ]:
# Célula Code 9: Testes específicos para documentos jurídicos brasileiros

def detectar_estrutura_juridica_cortada(chunk_text):
    """
    Detecta se um chunk cortou no meio de:
    - Artigo de lei
    - Inciso
    - Alínea
    - Parágrafo
    - Ementa
    
    Retorna: (tem_estrutura_cortada: bool, detalhes: str)
    """
    
    # Padrões jurídicos brasileiros
    padroes = {
        'artigo_aberto': r'Art\.\s+\d+[^.]*$',  # Artigo sem ponto final
        'inciso_aberto': r'[IVX]+ -[^.]*$',      # Inciso sem ponto final
        'aliena_aberta': r'[a-z]\)[^.]*$',       # Alínea sem ponto final
        'paragrafo_aberto': r'§ \d+[^.]*$',      # Parágrafo sem ponto final
    }
    
    detalhes = []
    
    # Remover quebras de linha para análise
    ultima_linha = chunk_text.strip().split('\n')[-1]
    
    for tipo, padrao in padroes.items():
        if re.search(padrao, ultima_linha):
            detalhes.append(tipo.replace('_', ' '))
    
    return len(detalhes) > 0, ', '.join(detalhes) if detalhes else 'Nenhum'

# Analisar todos os chunks
resultados_juridicos = []

for estrategia, chunks in chunks_por_estrategia.items():
    chunks_com_corte = 0
    
    for chunk in chunks:
        tem_corte, tipo_corte = detectar_estrutura_juridica_cortada(chunk.page_content)
        
        if tem_corte:
            chunks_com_corte += 1
        
        resultados_juridicos.append({
            'estrategia': estrategia,
            'chunk_id': chunk.metadata['chunk_id'],
            'tem_estrutura_cortada': tem_corte,
            'tipo_corte': tipo_corte,
        })
    
    pct_corte = (chunks_com_corte / len(chunks) * 100) if chunks else 0
    print(f"{estrategia:<12} | {chunks_com_corte:>2}/{len(chunks):>2} chunks com estrutura cortada ({pct_corte:>5.1f}%)")

df_juridicos = pd.DataFrame(resultados_juridicos)

print("\n📋 Resumo por tipo de estrutura cortada:\n")
for tipo in df_juridicos[df_juridicos['tem_estrutura_cortada']]['tipo_corte'].unique():
    count = len(df_juridicos[df_juridicos['tipo_corte'].str.contains(tipo, na=False)])
    print(f"  {tipo}: {count} ocorrências")

print("\n💡 Conclusão para contexto jurídico:")
for est in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    pct = (len(df_juridicos[(df_juridicos['estrategia'] == est) & (df_juridicos['tem_estrutura_cortada'])])
           / len(df_juridicos[df_juridicos['estrategia'] == est]) * 100)
    if pct > 20:
        print(f"  ⚠️  {est}: {pct:.1f}% chunks com estrutura cortada - NÃO RECOMENDADO")
    else:
        print(f"  ✅ {est}: {pct:.1f}% chunks com estrutura cortada - ACEITÁVEL")

## Seção: Visualização UMAP dos Embeddings

In [ ]:
# Célula Code 10: Gerar embeddings e UMAP para visualização

from umap import UMAP
import matplotlib.pyplot as plt
import numpy as np

print("🔄 Gerando embeddings para visualização UMAP...")

# Coletar todos os chunks com seus embeddings
todos_chunks = []
todos_embeddings = []
todos_labels = []
todos_cores = []

mapa_cores = {'FIXED': 0, 'RECURSIVE': 1, 'SEMANTIC': 2}

for estrategia, chunks in chunks_por_estrategia.items():
    print(f"  {estrategia}...", end="")
    
    for chunk in chunks:
        # Gerar embedding (usar modelo já carregado)
        embedding = model_embedding.encode(chunk.page_content[:300], convert_to_numpy=True)  # Truncar para velocidade
        
        todos_chunks.append(chunk)
        todos_embeddings.append(embedding)
        todos_labels.append(estrategia)
        todos_cores.append(mapa_cores[estrategia])
    
    print(" ✅")

# Converter para array numpy
todos_embeddings = np.array(todos_embeddings)

print(f"\n✅ {len(todos_embeddings)} embeddings gerados!")
print(f"   Dimensão de cada embedding: {todos_embeddings.shape[1]}")

# Aplicar UMAP para redução a 2D
print("\n🔄 Aplicando UMAP para redução dimensional...")
umap_model = UMAP(n_components=2, metric='cosine', random_state=42, n_neighbors=15, min_dist=0.1)
umap_embeddings = umap_model.fit_transform(todos_embeddings)

print("✅ UMAP completo!")

# Criar visualização
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Análise de Embeddings via UMAP', fontsize=16, fontweight='bold')

# Plot 1: Colorir por estratégia
ax1 = axes[0]
for estrategia in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    mask = np.array(todos_labels) == estrategia
    ax1.scatter(umap_embeddings[mask, 0], umap_embeddings[mask, 1],
               label=estrategia, alpha=0.6, s=50)
ax1.set_xlabel('UMAP 1')
ax1.set_ylabel('UMAP 2')
ax1.set_title('Chunks coloridos por estratégia')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Colorir por score de coerência
ax2 = axes[1]
# Obter scores de coerência para cada chunk
scores_coerencia_lista = []
for chunk in todos_chunks:
    est = chunk.metadata['strategy']
    chunk_id = chunk.metadata['chunk_id']
    score = df_coerencia[
        (df_coerencia['estrategia'] == est) & (df_coerencia['chunk_id'] == chunk_id)
    ]['coerencia'].values
    scores_coerencia_lista.append(score[0] if len(score) > 0 else 0.5)

scores_coerencia_array = np.array(scores_coerencia_lista)
scatter = ax2.scatter(umap_embeddings[:, 0], umap_embeddings[:, 1],
                     c=scores_coerencia_array, cmap='RdYlGn', s=50, alpha=0.7)
ax2.set_xlabel('UMAP 1')
ax2.set_ylabel('UMAP 2')
ax2.set_title('Chunks coloridos por score de coerência')
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Coerência')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/umap_chunks.png', dpi=100, bbox_inches='tight')
print("\n✅ Gráfico UMAP salvo em /tmp/umap_chunks.png")
plt.show()

print("\n💡 Interpretação do UMAP:")
print("  - Pontos PRÓXIMOS: chunks semanticamente similares")
print("  - Pontos AFASTADOS: chunks sobre temas diferentes")
print("  - Cores VERMELHAS: chunks menos coerentes (cuidado!)")
print("  - Cores VERDES: chunks muito coerentes (ótimo!)")

## Seção: Conclusões e Boas Práticas

In [ ]:
# Célula Code 11: Gerar relatório final automático

print("="*80)
print("RELATÓRIO FINAL: ANÁLISE QUALITATIVA DE CHUNKS")
print("="*80)

print("\n🎯 RESUMO EXECUTIVO")
print("-" * 80)

melhor = df_score_final.loc[df_score_final['score_10'].idxmax()]
print(f"Melhor estratégia: {melhor['estrategia'].values[0]}")
print(f"Score: {melhor['score_10'].values[0]}/10")

print("\n📊 SCORES FINAIS")
print("-" * 80)
for _, row in df_score_final.iterrows():
    est = row['estrategia']
    score = row['score_10']
    barra = "█" * int(score) + "░" * (10 - int(score))
    print(f"  {est:<12} [{barra}] {score:.1f}/10")

print("\n🔍 ANÁLISE POR CRITÉRIO")
print("-" * 80)

print("\n1️⃣  COERÊNCIA SEMÂNTICA (peso 0.35)")
for _, row in df_coerencia.groupby('estrategia')['coerencia'].agg(['mean', 'std']).iterrows():
    print(f"  {row.name:<12} | Média: {row['mean']:.3f} | Desvio: {row['std']:.3f}")
print("  ✅ Expectativa: > 0.6")

print("\n2️⃣  INDEPENDÊNCIA CONTEXTUAL (peso 0.30)")
for est in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    orfaos = len(df_independencia[(df_independencia['estrategia'] == est) & (df_independencia['eh_orfao'])])
    total = len(df_independencia[df_independencia['estrategia'] == est])
    pct = (orfaos / total * 100) if total > 0 else 0
    print(f"  {est:<12} | {pct:.1f}% chunks órfãos")
print("  ✅ Expectativa: < 10%")

print("\n3️⃣  DISTRIBUIÇÃO DE TAMANHOS (peso 0.20)")
for _, row in df_tamanhos.iterrows():
    cv = row['cv']
    qualidade = "✅ Excelente" if cv < 0.3 else "⚠️  Aceitável" if cv < 0.6 else "❌ Ruim"
    print(f"  {row['estrategia']:<12} | CV: {cv:.3f} {qualidade}")

print("\n4️⃣  COMPLETUDE INFORMACIONAL (peso 0.15)")
for est in ['FIXED', 'RECURSIVE', 'SEMANTIC']:
    if len(df_completude[df_completude['estrategia'] == est]) > 0:
        avg = df_completude[df_completude['estrategia'] == est]['score_completude'].mean()
        print(f"  {est:<12} | Score: {avg:.3f}")

print("\n\n📋 RECOMENDAÇÕES POR TIPO DE DOCUMENTO")
print("-" * 80)
print("""
  📄 Para LEIS e CÓDIGOS JURÍDICOS:
     → Use RECURSIVE chunking
     → Respeitará artigos, incisos, alíneas
     → Chunks completos e independentes
  
  📄 Para JURISPRUDÊNCIA (Acórdãos, decisões):
     → Use SEMANTIC chunking
     → Agrupa por tema da decisão
     → Mantém contexto argumentativo
  
  📄 Para RELATÓRIOS e ANÁLISES:
     → Use FIXED ou SEMANTIC
     → Bom para estruturas homogêneas
     → Cuidado com tabelas!
""")

print("\n🚀 PRÓXIMOS PASSOS")
print("-" * 80)
print("""
  1. ✅ Você já AVALIOU a qualidade dos chunks!
  2. Próxima etapa: Escolher a melhor estratégia
  3. Indexar chunks escolhidos em OpenSearch (Lab 4)
  4. Criar RAG pipeline com retrieval + geração
  5. Monitorar qualidade de retrieval em produção
""")

print("\n" + "="*80)
print(f"Data do relatório: {pd.Timestamp.now().strftime('%d/%m/%Y %H:%M:%S')}")
print("="*80)

## Exercício Prático: Análise de Documento Próprio

**Objetivo**: Você trará um PDF jurídico ou de segurança pública e aplicará AS MESMAS 4 métricas de qualidade.

**Passos**:
1. Escolha um arquivo PDF de seu contexto
2. Converta para texto (use Docling do Lab 4)
3. Aplique as 3 estratégias de chunking (copie código das células anteriores)
4. Calcule os 4 critérios de qualidade
5. Compare qual estratégia é melhor para SEU documento
6. Documente suas conclusões

In [ ]:
# Célula Code 12: Template para exercício prático

# ========== EXERCÍCIO: Substitua com seu documento! ==========

# Passo 1: Carregar seu PDF (exemplo: upload ao Colab)
seu_documento = """
Cole aqui o conteúdo TEXT do seu documento PDF.
Remova qualquer cabeçalho/rodapé automático.
Mantenha a estrutura original (parágrafos, artigos, etc).
"""

# Passo 2: Aplicar as 3 estratégias (reutilizar funções definidas)
chunks_seu_doc_fixed = chunking_fixed(seu_documento, "seu_documento.txt")
chunks_seu_doc_recursive = chunking_recursive(seu_documento, "seu_documento.txt")
chunks_seu_doc_semantic = chunking_semantic_simple(seu_documento, "seu_documento.txt")

print(f"✅ Seu documento foi dividido em:")
print(f"  FIXED: {len(chunks_seu_doc_fixed)} chunks")
print(f"  RECURSIVE: {len(chunks_seu_doc_recursive)} chunks")
print(f"  SEMANTIC: {len(chunks_seu_doc_semantic)} chunks")

# Passo 3: Calcular coerência (reutilizar função)
print("\n🔄 Calculando coerência de seus chunks...")
resultados_seu_doc = []
for chunks in [chunks_seu_doc_fixed, chunks_seu_doc_recursive, chunks_seu_doc_semantic]:
    est = chunks[0].metadata['strategy'] if chunks else 'DESCONHECIDA'
    coerencias = [calcular_coerencia_chunk(c.page_content) for c in chunks]
    avg_coerencia = sum(coerencias) / len(coerencias) if coerencias else 0
    print(f"  {est}: coerência média = {avg_coerencia:.3f}")

print("\n✅ Exercício pronto! Repita o processo com métricas dos seus chunks.")
print("\n📝 Documente suas conclusões:")
print("   1. Qual estratégia foi melhor para SEU documento?")
print("   2. Qual foi o critério mais limitante?")
print("   3. Como melhorar a qualidade dos chunks?")